In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from google.colab import files

In [2]:

uploaded = files.upload()

df = pd.read_csv("listings.csv.gz")
df.head()

Saving listings.csv.gz to listings.csv.gz


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,90676,https://www.airbnb.com/rooms/90676,20250926165842,2025-09-26,city scrape,Short North - Italianate Cottage,Just steps from High Street and all the action...,The Short North Italianate Cottage is located ...,https://a0.muscache.com/pictures/950e43cd-53f3...,483306,...,4.87,4.92,4.76,2022-2475,f,3,3,0,0,5.11
1,591101,https://www.airbnb.com/rooms/591101,20250926165842,2025-09-26,city scrape,Bellows Studio Loft Apartment,Famous American artist George Bellows home wit...,A historic neighborhood of beautiful victorian...,https://a0.muscache.com/pictures/32b28442-ddf3...,2889677,...,4.91,4.89,4.89,2019-1230,f,1,0,1,0,2.14
2,927867,https://www.airbnb.com/rooms/927867,20250926165842,2025-09-26,city scrape,Full Private Room at the Hostel,The Wayfaring Buckeye Hostel is a social place...,We are located in the vibrant University Distr...,https://a0.muscache.com/pictures/08033ebe-286c...,4965048,...,4.85,4.65,4.68,2019-1314,f,5,1,4,0,0.56
3,1183297,https://www.airbnb.com/rooms/1183297,20250926165842,2025-09-26,city scrape,Hannah's Haus**Prime location in German Village**,Hannah's Haus in German Village is a stunning ...,German Village is a historic neighborhood just...,https://a0.muscache.com/pictures/miso/Hosting-...,6473080,...,4.98,4.78,4.82,NaN,f,3,3,0,0,1.80
4,1217678,https://www.airbnb.com/rooms/1217678,20250926165842,2025-09-26,city scrape,Comfortable rooms in Clintonville 1,"A cozy, warm, inviting place to stay in the he...",The house is on a quiet and residential street...,https://a0.muscache.com/pictures/airflow/Hosti...,5707733,...,4.99,4.97,4.94,2025-2824,f,2,0,2,0,1.90


In [3]:

df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)

df[['price']].head()

,price
0,128.0
1,112.0
2,105.0
3,253.0
4,74.0


In [4]:

df = df[df['price'] <= 1000].copy()

df['price'].describe()

,price
count,2676.000000
mean,135.128550
std,101.985291
min,23.000000
25%,78.000000
50%,108.000000
75%,159.000000
max,1000.000000


In [5]:

df['log_price'] = np.log1p(df['price'])

df[['price', 'log_price']].head()

,price,log_price
0,128.0,4.859812
1,112.0,4.727388
2,105.0,4.663439
3,253.0,5.537334
4,74.0,4.317488


In [6]:

df['bathrooms'] = df['bathrooms_text'].str.extract(r'(\d+\.?\d*)').astype(float)

df[['bathrooms_text', 'bathrooms']].head()

,bathrooms_text,bathrooms
0,2 baths,2.0
1,1 private bath,1.0
2,3 shared baths,3.0
3,2 baths,2.0
4,1.5 shared baths,1.5


In [7]:

features = [
    'accommodates',
    'bathrooms',
    'bedrooms',
    'beds',
    'minimum_nights',
    'number_of_reviews',
    'review_scores_rating',
    'room_type',
    'neighbourhood_cleansed'
]

df_model = df[features + ['log_price']].dropna()

df_model.head()

,accommodates,bathrooms,bedrooms,beds,minimum_nights,number_of_reviews,review_scores_rating,room_type,neighbourhood_cleansed,log_price
0,6,2.0,3.0,3.0,1,868,4.82,Entire home/apt,Near North/University,4.859812
1,2,1.0,1.0,1.0,2,342,4.93,Private room,Near East,4.727388
2,2,3.0,1.0,1.0,1,82,4.67,Private room,Near North/University,4.663439
3,6,2.0,3.0,3.0,30,87,4.92,Entire home/apt,Near South,5.537334
4,2,1.5,1.0,1.0,1,283,4.97,Private room,Clintonville,4.317488


In [8]:

df_model = pd.get_dummies(
    df_model,
    columns=['room_type', 'neighbourhood_cleansed'],
    drop_first=True
)

df_model.head()

,accommodates,bathrooms,bedrooms,beds,minimum_nights,number_of_reviews,review_scores_rating,log_price,room_type_Private room,room_type_Shared room,...,neighbourhood_cleansed_North Linden,neighbourhood_cleansed_Northeast,neighbourhood_cleansed_Northland,neighbourhood_cleansed_Northwest,neighbourhood_cleansed_Rocky Fork-Blacklick,neighbourhood_cleansed_South Linden,neighbourhood_cleansed_Southeast,neighbourhood_cleansed_West Olentangy,neighbourhood_cleansed_West Scioto,neighbourhood_cleansed_Westland
0,6,2.0,3.0,3.0,1,868,4.82,4.859812,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2,1.0,1.0,1.0,2,342,4.93,4.727388,True,False,...,False,False,False,False,False,False,False,False,False,False
2,2,3.0,1.0,1.0,1,82,4.67,4.663439,True,False,...,False,False,False,False,False,False,False,False,False,False
3,6,2.0,3.0,3.0,30,87,4.92,5.537334,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2,1.5,1.0,1.0,1,283,4.97,4.317488,True,False,...,False,False,False,False,False,False,False,False,False,False


In [9]:

X = df_model.drop(columns=['log_price'])
y = df_model['log_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Number of features:", X_train.shape[1])

Training rows: 1904
Test rows: 476
Number of features: 34


In [10]:

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [11]:

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 0.2680871466468521
RMSE: 0.32850410464069835
R²: 0.6669977392637971


In [12]:

results = pd.DataFrame({
    'Actual_Log_Price': y_test.values,
    'Predicted_Log_Price': y_pred
})

results.head(10)

,Actual_Log_Price,Predicted_Log_Price
0,5.198497,4.663665
1,4.094345,4.542102
2,4.934474,4.443587
3,4.762174,4.806549
4,4.454347,4.510203
5,4.595120,4.811438
6,5.843544,5.556116
7,5.648974,5.687990
8,3.988984,4.345077
9,4.955827,5.379928


In [13]:

import csv
from datetime import datetime

def log_experiment(run_name, features, preprocessing, model_name, hyperparameters, rmse, mae, r2, notes):
    with open("experiment_log.csv", mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([
            run_name,
            datetime.now().strftime("%Y-%m-%d"),
            "Thomas",
            "v1",
            features,
            preprocessing,
            model_name,
            hyperparameters,
            rmse,
            mae,
            r2,
            notes
        ])

In [14]:

log_experiment(
    run_name="baseline_lr_02_improved",
    features="accommodates+bathrooms+bedrooms+beds+minimum_nights+number_of_reviews+review_scores_rating+room_type+neighbourhood_cleansed",
    preprocessing="price cleaned + outliers removed (price <= 1000) + log_price created + bathrooms extracted + one-hot encoding + dropna",
    model_name="LinearRegression",
    hyperparameters="default",
    rmse=rmse,
    mae=mae,
    r2=r2,
    notes="Improved linear regression with outlier removal, log transform, and categorical features"
)

In [15]:

files.download("experiment_log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>